# Zh → Vi Transformer (trained from scratch)

- SentencePiece BPE per language, trained on the already word-segmented text (output keeps `_` and spacing exactly as `train.vi`).
- Pre-LN Transformer encoder-decoder, tied target embeddings, label smoothing, AMP fp16, warmup + cosine LR.
- Average of the last `AVG_LAST` epoch checkpoints, batched beam search with length penalty.
- Test sentences that appear verbatim in train get the train translation (`USE_LOOKUP`).

**Workflow**: run with `FULL_DATA = False` to read val BLEU, then `FULL_DATA = True` for the submission.
Runtime: GPU (T4). The dataset is embedded in the next cell, nothing to upload.

In [14]:
!nvidia-smi
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

Fri Oct 23 09:38:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             29W /   70W |    4931MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
# BTC dataset.zip (train / public_test / private_test), base64-embedded
DATASET_B64 = '...'

In [16]:
import os, sys, subprocess, io, zipfile, base64
try:
    import sentencepiece, sacrebleu
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'sacrebleu'], check=True)

WORK = '/content/mt' if os.path.isdir('/content') else os.path.abspath('mt_work')
OUT_DIR = f'{WORK}/out'
os.makedirs(OUT_DIR, exist_ok=True)
DATA = f'{WORK}/dataset'
if not os.path.isdir(DATA):
    zipfile.ZipFile(io.BytesIO(base64.b64decode(DATASET_B64))).extractall(WORK)
print('data:', sorted(os.listdir(DATA)), '| outputs ->', OUT_DIR)
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True)

data: ['private_test', 'public_test', 'train'] | outputs -> /content/mt/out


CompletedProcess(args='nvidia-smi --query-gpu=name,memory.total --format=csv', returncode=0)

## Config

In [17]:
FULL_DATA = False        # True: train on 100% of train (no val) for the final submission
RESUME = False           # True only to continue after a Colab disconnect; False always trains from scratch
VAL_FRAC = 0.05
SEED = 42

VOCAB_ZH = 8000
VOCAB_VI = 8000
D_MODEL, N_HEAD, D_FF = 512, 8, 2048
N_ENC, N_DEC = 4, 4
DROPOUT = 0.3

EPOCHS = 60
BATCH_TOKENS = 3000      # max (padded) tokens per batch
LR = 5e-4
WARMUP = 1000            # steps
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
BPE_DROPOUT = 0.1        # BPE-dropout on the training set, re-sampled every epoch (0 disables); val/test use plain BPE
EVAL_EVERY = 2           # epochs between val BLEU checks (greedy)
AVG_LAST = 5             # average the last N epoch checkpoints

BEAM = 5
LEN_ALPHA = 0.6          # GNMT length penalty
MAX_LEN = 100            # subword cap per sentence
USE_LOOKUP = True        # copy the train translation for test sentences seen verbatim in train
SUB_CSV = 'nlp_submission.csv'      # file name inside the zip (required by the scorer)

if os.environ.get('MT_SMOKE') == '1':   # local CPU smoke test
    D_MODEL, N_HEAD, D_FF, N_ENC, N_DEC = 64, 4, 128, 1, 1
    EPOCHS, EVAL_EVERY, AVG_LAST, WARMUP, BEAM = 3, 1, 2, 50, 3
    VOCAB_ZH = VOCAB_VI = 4000

RUN_NAME = f'tf_{"full" if FULL_DATA else "val"}' + ('_bpedrop' if BPE_DROPOUT else '')
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(RUN_NAME, DEVICE)

tf_val_bpedrop cuda


## Data

In [18]:
import random
from collections import Counter, defaultdict

def read_lines(path):
    with open(path, encoding='utf-8') as f:
        lines = f.read().replace('\r', '').split('\n')
    if lines and lines[-1] == '':
        lines = lines[:-1]
    return [' '.join(l.split()) for l in lines]

train_zh = read_lines(f'{DATA}/train/train.zh')
train_vi = read_lines(f'{DATA}/train/train.vi')
pub_zh = read_lines(f'{DATA}/public_test/public_test.zh')
priv_zh = read_lines(f'{DATA}/private_test/private_test.zh')
assert len(train_zh) == len(train_vi)
pairs = [(z, v) for z, v in zip(train_zh, train_vi) if z and v]
print(f'train pairs {len(pairs)} | public {len(pub_zh)} | private {len(priv_zh)}')

rng = random.Random(SEED)
idx = list(range(len(pairs)))
rng.shuffle(idx)
n_val = 0 if FULL_DATA else int(len(pairs) * VAL_FRAC)
val_pairs = [pairs[i] for i in idx[:n_val]]
trn_pairs = [pairs[i] for i in idx[n_val:]]
print(f'train {len(trn_pairs)} | val {len(val_pairs)}')

def build_lookup(ps):
    c = defaultdict(Counter)
    for z, v in ps:
        c[z][v] += 1
    return {z: cnt.most_common(1)[0][0] for z, cnt in c.items()}

lookup = build_lookup(trn_pairs)
for name, xs in [('val', [z for z, _ in val_pairs]), ('public', pub_zh), ('private', priv_zh)]:
    if xs:
        print(f'{name}: {sum(x in lookup for x in xs)}/{len(xs)} sentences seen verbatim in train split')

train pairs 32061 | public 1781 | private 1781
train 30458 | val 1603
val: 147/1603 sentences seen verbatim in train split
public: 165/1781 sentences seen verbatim in train split
private: 156/1781 sentences seen verbatim in train split


In [19]:
import sentencepiece as spm
PAD, UNK, BOS, EOS = 0, 1, 2, 3

def train_spm(lines, prefix, vocab):
    tmp = prefix + '.txt'
    with open(tmp, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    spm.SentencePieceTrainer.Train(
        input=tmp, model_prefix=prefix, vocab_size=vocab, model_type='bpe',
        character_coverage=1.0, normalization_rule_name='identity',
        pad_id=PAD, unk_id=UNK, bos_id=BOS, eos_id=EOS, minloglevel=2)
    return spm.SentencePieceProcessor(model_file=prefix + '.model')

sp_zh = train_spm([z for z, _ in trn_pairs], f'{WORK}/spm_zh_{RUN_NAME}', VOCAB_ZH)
sp_vi = train_spm([v for _, v in trn_pairs], f'{WORK}/spm_vi_{RUN_NAME}', VOCAB_VI)

def enc_src(s): return sp_zh.encode(s)[:MAX_LEN - 1] + [EOS]
def enc_tgt(s): return sp_vi.encode(s)[:MAX_LEN - 1]

def encode_train(sample):
    kw = dict(enable_sampling=True, alpha=BPE_DROPOUT, nbest_size=-1) if sample and BPE_DROPOUT else {}
    src = [sp_zh.encode(z, **kw)[:MAX_LEN - 1] + [EOS] for z, _ in trn_pairs]
    tgt = [sp_vi.encode(v, **kw)[:MAX_LEN - 1] for _, v in trn_pairs]
    return src, tgt

trn_src, trn_tgt = encode_train(False)
v0 = trn_pairs[0][1]
print('vi roundtrip:', repr(sp_vi.decode(sp_vi.encode(v0))), '==', repr(v0), sp_vi.decode(sp_vi.encode(v0)) == v0)
print('avg subwords zh %.1f vi %.1f' % (sum(map(len, trn_src)) / len(trn_src), sum(map(len, trn_tgt)) / len(trn_tgt)))
if BPE_DROPOUT:
    s, t = encode_train(True)
    print('with BPE-dropout: zh %.1f vi %.1f' % (sum(map(len, s)) / len(s), sum(map(len, t)) / len(t)))
    print('example:', [sp_vi.id_to_piece(i) for i in t[0]])

vi roundtrip: 'Vâng , tôi đang tìm một món quà cho chị tôi .' == 'Vâng , tôi đang tìm một món quà cho chị tôi .' True
avg subwords zh 8.8 vi 10.8
with BPE-dropout: zh 10.0 vi 13.3
example: ['▁V', 'â', 'ng', '▁,', '▁t', 'ôi', '▁đang', '▁tìm', '▁một', '▁món', '▁quà', '▁cho', '▁chị', '▁tôi', '▁', '.']


In [20]:
def make_batches(src, tgt, max_tokens, shuffle):
    jitter = (lambda: random.random()) if shuffle else (lambda: 0.0)
    order = sorted(range(len(src)), key=lambda i: max(len(src[i]), len(tgt[i]) + 1) + jitter())
    batches, cur, mx = [], [], 0
    for i in order:
        l = max(len(src[i]), len(tgt[i]) + 1)
        if cur and max(mx, l) * (len(cur) + 1) > max_tokens:
            batches.append(cur); cur, mx = [], 0
        cur.append(i); mx = max(mx, l)
    if cur:
        batches.append(cur)
    if shuffle:
        random.shuffle(batches)
    return batches

def pad(seqs):
    out = torch.full((len(seqs), max(map(len, seqs))), PAD, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = torch.tensor(s)
    return out

def collate(ids):
    src = pad([trn_src[i] for i in ids])
    tgt_in = pad([[BOS] + trn_tgt[i] for i in ids])
    tgt_out = pad([trn_tgt[i] + [EOS] for i in ids])
    return src, tgt_in, tgt_out

print('batches/epoch:', len(make_batches(trn_src, trn_tgt, BATCH_TOKENS, False)))

batches/epoch: 124


## Model

In [21]:
import math
import torch.nn as nn
import torch.nn.functional as F

class Seq2SeqTF(nn.Module):
    def __init__(self, v_src, v_tgt):
        super().__init__()
        d = D_MODEL
        self.scale = math.sqrt(d)
        self.src_emb = nn.Embedding(v_src, d, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(v_tgt, d, padding_idx=PAD)
        for e in (self.src_emb, self.tgt_emb):
            nn.init.normal_(e.weight, std=d ** -0.5)
            nn.init.zeros_(e.weight[PAD])
        pos = torch.arange(512).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe = torch.zeros(512, d)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer('pe', pe, persistent=False)
        self.drop = nn.Dropout(DROPOUT)
        enc_layer = nn.TransformerEncoderLayer(d, N_HEAD, D_FF, DROPOUT, batch_first=True, norm_first=True)
        dec_layer = nn.TransformerDecoderLayer(d, N_HEAD, D_FF, DROPOUT, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, N_ENC, norm=nn.LayerNorm(d), enable_nested_tensor=False)
        self.decoder = nn.TransformerDecoder(dec_layer, N_DEC, norm=nn.LayerNorm(d))
        self.out = nn.Linear(d, v_tgt, bias=False)
        self.out.weight = self.tgt_emb.weight

    def embed(self, emb, x):
        return self.drop(emb(x) * self.scale + self.pe[:x.size(1)])

    def encode(self, src):
        mask = src == PAD
        return self.encoder(self.embed(self.src_emb, src), src_key_padding_mask=mask), mask

    def decode(self, tgt_in, mem, mem_mask):
        T = tgt_in.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), 1)
        h = self.decoder(self.embed(self.tgt_emb, tgt_in), mem, tgt_mask=causal,
                         memory_key_padding_mask=mem_mask)
        return self.out(h)

    def forward(self, src, tgt_in):
        mem, mask = self.encode(src)
        return self.decode(tgt_in, mem, mask)

def new_model():
    return Seq2SeqTF(sp_zh.get_piece_size(), sp_vi.get_piece_size()).to(DEVICE)

model = new_model()
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params')

37.6M params


## Decoding (batched greedy / beam search)

In [22]:
import sacrebleu

@torch.no_grad()
def greedy_batch(m, src):
    mem, mask = m.encode(src)
    B = src.size(0)
    ys = torch.full((B, 1), BOS, dtype=torch.long, device=src.device)
    done = torch.zeros(B, dtype=torch.bool, device=src.device)
    for _ in range(int(src.size(1) * 1.5) + 10):
        nxt = m.decode(ys, mem, mask)[:, -1].argmax(-1)
        nxt = nxt.masked_fill(done, PAD)
        ys = torch.cat([ys, nxt[:, None]], 1)
        done |= nxt == EOS
        if done.all():
            break
    return ys[:, 1:].tolist()

@torch.no_grad()
def beam_batch(m, src, K=BEAM, alpha=LEN_ALPHA):
    B, S = src.shape
    dev = src.device
    mem, mask = m.encode(src)
    mem = mem.repeat_interleave(K, 0)
    mask = mask.repeat_interleave(K, 0)
    ys = torch.full((B * K, 1), BOS, dtype=torch.long, device=dev)
    scores = torch.full((B, K), -1e9, device=dev)
    scores[:, 0] = 0.0
    done = torch.zeros(B * K, dtype=torch.bool, device=dev)
    lengths = torch.zeros(B * K, device=dev)
    ar = torch.arange(B, device=dev)[:, None] * K
    for _ in range(int(S * 1.5) + 10):
        logp = m.decode(ys, mem, mask)[:, -1].float().log_softmax(-1)
        V = logp.size(-1)
        # finished beams can only extend with PAD at no cost
        logp[done] = -1e9
        logp[done, PAD] = 0.0
        cand = (scores.view(-1, 1) + logp).view(B, K * V)
        scores, flat = cand.topk(K, -1)
        beam = (flat // V + ar).view(-1)
        tok = (flat % V).view(-1)
        ys = torch.cat([ys[beam], tok[:, None]], 1)
        lengths = lengths[beam] + (~done[beam]).float()
        done = done[beam] | (tok == EOS)
        if done.all():
            break
    lp = ((5 + lengths) / 6) ** alpha
    best = (scores.view(-1) / lp).view(B, K).argmax(-1) + ar[:, 0]
    return ys[best, 1:].tolist()

def ids_to_text(ids):
    if EOS in ids:
        ids = ids[:ids.index(EOS)]
    return sp_vi.decode([i for i in ids if i > EOS])

def translate(m, sents, beam=0, batch=64):
    m.eval()
    srcs = [enc_src(s) for s in sents]
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    out = [None] * len(srcs)
    for b in range(0, len(order), batch):
        ids = order[b:b + batch]
        src = pad([srcs[i] for i in ids]).to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            res = beam_batch(m, src) if beam else greedy_batch(m, src)
        for i, r in zip(ids, res):
            out[i] = ids_to_text(r)
    return out

def bleu(hyps, refs):
    return sacrebleu.corpus_bleu(hyps, [refs], force=True).score

def apply_lookup(srcs, hyps, table):
    return [table.get(s, h) for s, h in zip(srcs, hyps)]

## Train

In [23]:
import time, json, glob

random.seed(SEED); torch.manual_seed(SEED)
model = new_model()
decay = [p for n, p in model.named_parameters() if p.dim() > 1]
no_decay = [p for n, p in model.named_parameters() if p.dim() <= 1]
opt = torch.optim.AdamW([{'params': decay, 'weight_decay': WEIGHT_DECAY},
                         {'params': no_decay, 'weight_decay': 0.0}], lr=LR, betas=(0.9, 0.98), eps=1e-9)
steps_per_epoch = max(len(make_batches(*encode_train(True), BATCH_TOKENS, True)) for _ in range(3))
TOTAL = EPOCHS * steps_per_epoch

def lr_at(step):
    if step < WARMUP:
        return (step + 1) / WARMUP
    t = min(1.0, (step - WARMUP) / max(1, TOTAL - WARMUP))
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * t))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
scaler = torch.amp.GradScaler(enabled=DEVICE.type == 'cuda')
crit = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=LABEL_SMOOTH)

val_src = [z for z, _ in val_pairs]
val_ref = [v for _, v in val_pairs]
history, start = [], 1
last_path = f'{OUT_DIR}/{RUN_NAME}_last.pt'
if RESUME and os.path.exists(last_path):
    ck = torch.load(last_path, map_location=DEVICE)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
    history, start = ck['history'], ck['epoch'] + 1
    print('resumed from epoch', ck['epoch'])
else:
    print(f'{RUN_NAME}: training from scratch, {len(trn_pairs)} pairs, {EPOCHS} epochs')

for ep in range(start, EPOCHS + 1):
    model.train()
    t0, tot, ntok = time.time(), 0.0, 0
    if BPE_DROPOUT:
        trn_src, trn_tgt = encode_train(True)   # fresh segmentation each epoch (collate reads these)
    for ids in make_batches(trn_src, trn_tgt, BATCH_TOKENS, True):
        src, tgt_in, tgt_out = (x.to(DEVICE, non_blocking=True) for x in collate(ids))
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            logits = model(src, tgt_in)
        loss = crit(logits.float().view(-1, logits.size(-1)), tgt_out.view(-1))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        n = (tgt_out != PAD).sum().item()
        tot += loss.item() * n; ntok += n
    rec = {'epoch': ep, 'loss': tot / ntok, 'lr': sched.get_last_lr()[0], 'sec': time.time() - t0}
    if val_pairs and (ep % EVAL_EVERY == 0 or ep == EPOCHS):
        rec['val_bleu'] = bleu(translate(model, val_src), val_ref)
    history.append(rec)
    print(' | '.join(f'{k} {v:.4g}' if isinstance(v, float) else f'{k} {v}' for k, v in rec.items()), flush=True)
    if ep > EPOCHS - AVG_LAST:
        torch.save(model.state_dict(), f'{OUT_DIR}/{RUN_NAME}_ep{ep}.pt')
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'scaler': scaler.state_dict(), 'epoch': ep, 'history': history}, last_path)
json.dump(history, open(f'{OUT_DIR}/history_{RUN_NAME}.json', 'w'), indent=1)

tf_val_bpedrop: training from scratch, 30458 pairs, 60 epochs
epoch 1 | loss 7.68 | lr 7.55e-05 | sec 15.21
epoch 2 | loss 5.873 | lr 0.0001505 | sec 15.41 | val_bleu 4.194
epoch 3 | loss 5.204 | lr 0.0002255 | sec 15.31
epoch 4 | loss 4.751 | lr 0.0003005 | sec 15.39 | val_bleu 9.958
epoch 5 | loss 4.446 | lr 0.0003755 | sec 15.74
epoch 6 | loss 4.183 | lr 0.0004505 | sec 15.85 | val_bleu 16.07
epoch 7 | loss 3.984 | lr 0.0005 | sec 15.92
epoch 8 | loss 3.811 | lr 0.0004992 | sec 16.21 | val_bleu 20.84
epoch 9 | loss 3.663 | lr 0.0004977 | sec 15.95
epoch 10 | loss 3.544 | lr 0.0004953 | sec 15.9 | val_bleu 24.29
epoch 11 | loss 3.452 | lr 0.0004921 | sec 15.8
epoch 12 | loss 3.363 | lr 0.000488 | sec 16.09 | val_bleu 26.74
epoch 13 | loss 3.29 | lr 0.0004831 | sec 15.89
epoch 14 | loss 3.234 | lr 0.0004775 | sec 16.06 | val_bleu 29.24
epoch 15 | loss 3.17 | lr 0.0004711 | sec 16.19
epoch 16 | loss 3.117 | lr 0.0004639 | sec 15.9 | val_bleu 31.12
epoch 17 | loss 3.073 | lr 0.000456 | 

## Checkpoint averaging + val BLEU

In [24]:
paths = [f'{OUT_DIR}/{RUN_NAME}_ep{e}.pt' for e in range(EPOCHS - AVG_LAST + 1, EPOCHS + 1)]
paths = [p for p in paths if os.path.exists(p)]
avg = None
for p in paths:
    sd = torch.load(p, map_location='cpu')
    avg = {k: v.float().clone() for k, v in sd.items()} if avg is None else {k: avg[k] + sd[k].float() for k in avg}
avg_model = new_model()
avg_model.load_state_dict({k: v / len(paths) for k, v in avg.items()})
print(f'averaged {len(paths)} checkpoints')

if val_pairs:
    t0 = time.time()
    res = {
        'last greedy': bleu(translate(model, val_src), val_ref),
        'avg greedy': bleu(translate(avg_model, val_src), val_ref),
    }
    val_hyp = translate(avg_model, val_src, beam=BEAM)
    res[f'avg beam{BEAM}'] = bleu(val_hyp, val_ref)
    res[f'avg beam{BEAM} + lookup'] = bleu(apply_lookup(val_src, val_hyp, lookup), val_ref)
    for k, v in res.items():
        print(f'{k:>22}: {v:.2f}')
    print(f'({time.time() - t0:.0f}s)')
    with open(f'{OUT_DIR}/val_{RUN_NAME}.tsv', 'w', encoding='utf-8') as f:
        for z, r, h in zip(val_src, val_ref, val_hyp):
            f.write(f'{z}\t{r}\t{h}\n')
    for z, r, h in list(zip(val_src, val_ref, val_hyp))[:8]:
        print(f'\nZH : {z}\nREF: {r}\nHYP: {h}')

averaged 5 checkpoints
           last greedy: 39.63
            avg greedy: 39.41
             avg beam5: 40.39
    avg beam5 + lookup: 42.72
(19s)

ZH : 这个 巴士 去 联合 广场 的 度假 旅馆 吗 ？
REF: Xe_buýt này có đi đến Quảng_trường Holiday_Inn_Union không ?
HYP: Xe_buýt này có đi đến khách_sạn Quảng_trường Thống_nhất không ?

ZH : 回 日本 后 我们 会 小心 的 继续 发展 。
REF: Chúng_tôi sẽ theo_dõi tiến_triển một_cách cẩn_thận sau khi chúng_tôi đã trở_về Nhật_Bản .
HYP: Sau khi về Nhật_Bản chúng_tôi sẽ cẩn_thận sự phát_triển của chúng_ta .

ZH : 给 我 一 份 面包 煎蛋 。
REF: cho tôi 1 phần bánh_mì ốp_la .
HYP: Làm_ơn cho tôi một trứng ốp_la .

ZH : 十月 十日 的 健康 体育节 是 为了 促进 人民 的 健康 而 设立 的 法定 节假日 。
REF: Ngày Sức khỏe - Thể_Thao ngày 10 tháng 10 là ngày lễ quốc_gia được đặt ra để nâng_cao sức_khoẻ của người dân .
HYP: Ngày lễ thể_dục của ngày 10 tháng 10 là vì tình_yêu của nhân_viên quốc_tế vào ngày lễ quốc_tế .

ZH : 我 耳鸣 。
REF: Tôi bị ù tai .
HYP: Tôi bị mặt_mặt .

ZH : 我 把 钱 放进 了 机器 可是 没有 票 。
REF: Tôi bỏ tiền vào máy , nhưn

## Predict test + submission zip

In [25]:
import pandas as pd
test_zh = pub_zh + priv_zh
t0 = time.time()
test_hyp = translate(avg_model, test_zh, beam=BEAM)
full_lookup = build_lookup(pairs)
if USE_LOOKUP:
    print('lookup replaced', sum(z in full_lookup for z in test_zh), 'of', len(test_zh))
    test_hyp = apply_lookup(test_zh, test_hyp, full_lookup)
print(f'translated {len(test_zh)} in {time.time() - t0:.0f}s, empty outputs: {sum(not h for h in test_hyp)}')

# scorer reads only the private test, row-aligned with private_test.zh
sub = pd.DataFrame({'tieng_trung': priv_zh, 'tieng_viet': test_hyp[len(pub_zh):]})
csv_path = f'{OUT_DIR}/{SUB_CSV}'
sub.to_csv(csv_path, index=False, encoding='utf-8-sig')
zip_path = f'{OUT_DIR}/nlp_submission_{RUN_NAME}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname=SUB_CSV)
print(zip_path, os.path.getsize(zip_path) // 1024, 'KB')
print(sub.sample(8, random_state=0).to_string())

lookup replaced 333 of 3562
translated 3562 in 25s, empty outputs: 0
/content/mt/out/nlp_submission_tf_val_bpedrop.zip 53 KB
                    tieng_trung                                                    tieng_viet
1337               这些 耳环 多少 钱 ？                           Những bông_tai này bao_nhiêu tiền ?
1411                   你 的 名字 ？                       Tôi có_thể hỏi tên của bạn được không ?
1182            你 什么 时候 出院 回家 ？                                          Bạn khi nào về nhà ?
962   请 帮 我 找 个 戴 顶 红 帽子 的 男孩 。                     Làm_ơn giúp tôi tìm một chiếc mũ màu đỏ .
1739      越南人 准备 过年 大概 也 象 这样 。  Người việt nam chuẩn_bị ăn tết đại_khái cũng giống như thế .
1541                         袖口                                                        tay_áo
161      有 从 芝加哥 去 西雅图 的 航班 吗 ？                          Có chuyến bay nào đi Seattle không ?
141      我 恐怕 所有 的 座位 都 订 满 了 。                Tôi e rằng tất_cả chỗ ngồi đã được đặt trước .


## Bundle (lets the results be recovered from this notebook file)

In [26]:
import tarfile
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode='w:gz') as tf:
    for p in [zip_path, f'{OUT_DIR}/history_{RUN_NAME}.json', f'{OUT_DIR}/val_{RUN_NAME}.tsv']:
        if os.path.exists(p):
            tf.add(p, arcname=os.path.basename(p))
b64 = base64.b64encode(buf.getvalue()).decode()
print(f'BUNDLE_BEGIN {RUN_NAME} {len(b64)}')
print(b64)
print('BUNDLE_END')

BUNDLE_BEGIN tf_val_bpedrop 172652
...
BUNDLE_END
